# Day 4 - Plotly: Daily NAV with market regime highlights (2022-2026)

This notebook plots **daily NAV** for the 40 schemes using:
- `Data/processed/fund_master_clean.csv`
- `Data/processed/nav_history_clean.csv`

**Highlights**:
- **Bull run**: calendar year **2023**
- **Corrections (2024)**: detected via drawdown from a trailing **90-day** peak exceeding **8%**


In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

# --- Repo-root detection (so notebook works from any working directory) ---
_HERE = Path(__file__).resolve() if '__file__' in globals() else Path.cwd()

def _find_repo_root(start: Path) -> Path:
    cand = start
    for _ in range(12):
        if (cand / 'Data' / 'processed' / 'nav_history_clean.csv').exists() and (cand / 'Data' / 'processed' / 'fund_master_clean.csv').exists():
            return cand
        if cand.name == 'notebooks':
            parent = cand.parent
            if (parent / 'Data' / 'processed' / 'nav_history_clean.csv').exists() and (parent / 'Data' / 'processed' / 'fund_master_clean.csv').exists():
                return parent
        cand = cand.parent
    return start.parent

_REPO_ROOT = _find_repo_root(_HERE)
DATA_DIR = _REPO_ROOT / 'Data' / 'processed'

nav_path = DATA_DIR / 'nav_history_clean.csv'
fund_path = DATA_DIR / 'fund_master_clean.csv'

if not nav_path.exists():
    raise FileNotFoundError(f'Missing file: {nav_path.resolve()}')
if not fund_path.exists():
    raise FileNotFoundError(f'Missing file: {fund_path.resolve()}')

nav_df = pd.read_csv(nav_path)
fund_df = pd.read_csv(fund_path)

nav_df['date'] = pd.to_datetime(nav_df['date'], errors='coerce')
nav_df['amfi_code'] = pd.to_numeric(nav_df['amfi_code'], errors='coerce').astype('Int64')
nav_df['nav'] = pd.to_numeric(nav_df['nav'], errors='coerce')

fund_df['amfi_code'] = pd.to_numeric(fund_df['amfi_code'], errors='coerce').astype('Int64')
fund_df = fund_df.dropna(subset=['amfi_code', 'scheme_name']).copy()

codes_sorted = fund_df.sort_values('amfi_code')['amfi_code'].astype(int).tolist()
amfi_to_name = dict(zip(fund_df['amfi_code'].astype(int), fund_df['scheme_name']))

START_DATE = pd.Timestamp('2022-01-01')
END_DATE = pd.Timestamp('2026-12-31')

# Correction detection parameters (Day 4)
PEAK_WINDOW_DAYS = 90
DRAWDOWN_THRESHOLD = 0.08  # 8%

def build_correction_mask(df_one: pd.DataFrame) -> np.ndarray:
    """Drawdown from trailing peak (lookback) exceeding threshold."""
    d = df_one.sort_values('date').copy()
    d = d.dropna(subset=['nav', 'date'])
    if d.empty:
        return np.zeros(len(df_one), dtype=bool)

    nav = d['nav'].astype(float)
    rolling_peak = nav.rolling(
        window=PEAK_WINDOW_DAYS,
        min_periods=max(2, PEAK_WINDOW_DAYS // 2)
    ).max()

    drawdown = (rolling_peak - nav) / rolling_peak
    return (drawdown > DRAWDOWN_THRESHOLD).fillna(False).to_numpy()

def contiguous_true_segments(mask: np.ndarray) -> list[tuple[int, int]]:
    segs: list[tuple[int, int]] = []
    in_seg = False
    start = 0
    for i, v in enumerate(mask):
        if v and not in_seg:
            in_seg = True
            start = i
        elif (not v) and in_seg:
            segs.append((start, i - 1))
            in_seg = False
    if in_seg:
        segs.append((start, len(mask) - 1))
    return segs

def plot_scheme_daily_nav(amfi_code: int, width: int = 1200, height: int = 450) -> go.Figure:
    df = nav_df.loc[nav_df['amfi_code'] == amfi_code].copy()
    df = df.loc[(df['date'] >= START_DATE) & (df['date'] <= END_DATE)].copy()
    df = df.sort_values('date')

    scheme_name = amfi_to_name.get(int(amfi_code), str(amfi_code))
    fig = go.Figure()

    if df.empty:
        fig.update_layout(title=f'{scheme_name} (AMFI {amfi_code}) - no data in range', width=width, height=height)
        return fig

    df = df.dropna(subset=['nav', 'date']).reset_index(drop=True)

    fig.add_trace(
        go.Scatter(
            x=df['date'],
            y=df['nav'],
            mode='lines',
            name='Daily NAV',
            line=dict(width=2)
        )
    )

    # Bull run band for 2023
    fig.add_vrect(
        x0=pd.Timestamp('2023-01-01'),
        x1=pd.Timestamp('2023-12-31'),
        fillcolor='green',
        opacity=0.08,
        layer='below',
        line_width=0,
        annotation_text='Bull run (2023)',
        annotation_position='top left',
    )

    # Corrections in 2024 via drawdown mask
    corr_mask_all = build_correction_mask(df)
    in_2024 = ((df['date'] >= pd.Timestamp('2024-01-01')) & (df['date'] <= pd.Timestamp('2024-12-31'))).to_numpy()
    corr_mask = corr_mask_all & in_2024
    corr_segs = contiguous_true_segments(corr_mask)

    for s, e in corr_segs:
        x0 = df.iloc[s]['date']
        x1 = df.iloc[e]['date']
        fig.add_vrect(x0=x0, x1=x1, fillcolor='red', opacity=0.10, layer='below', line_width=0)

    fig.update_layout(
        title=f'{scheme_name} (AMFI {amfi_code}) - Daily NAV',
        width=width,
        height=height,
        template='plotly_white',
        xaxis_title='Date',
        yaxis_title='NAV',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    )

    return fig

print('Loaded schemes:', len(codes_sorted))
print('DATA_DIR:', DATA_DIR.resolve())

Loaded schemes: 40
DATA_DIR: C:\Mutual Fund Analytics\Data\processed


In [2]:
# Quick validation: plot one scheme
fig = plot_scheme_daily_nav(codes_sorted[0])
fig.show()

## All schemes (optional)

Uncomment the loop below to render all schemes. This can take a while depending on your environment.

In [ ]:
# for i, amfi_code in enumerate(codes_sorted, start=1):
#     fig = plot_scheme_daily_nav(amfi_code)
#     fig.show()
#     if i >= len(codes_sorted):
#         break